# Plot the results of synthetic slip inversion based on ANY model of $\mu$ using the synthetic displacement based on ANY model of $\mu$ from a prescribed, ground-truth slip distribution on the fault

* **ANY** means the $\mu$ structure can be either homogeneous or heterogeneous.

* Once we know which elastic property the ground displacement is sensitive to, we may proceed with the inversion.

* The most basic thing to check is if you can recover the ground truth slip and fit the observation under a heterogeneous model of $\mu$. In theory, this should not be different from the homogeneous case, to make sure the code is running properly, you need to pass this test.

* The next level is how different the inferred slip distributions are under homogeneous and heterogeneous cases. The used ground displacement can be from either model, but maybe from heterogeneous $\mu$ structure should be more emphasized.


In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rc
from io import StringIO
import utils_plot as utp

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define the inversion results path
resultpath = "rst_locking/"

# Import GNSS data, originally from Feng et al. 2012, but no volcano sites, both trench-parallel and normal components
obs_disp_name = "data/CKfig6_data_final.csv"   # the EXACT data file for figure 6 in Kyriakopoulos et al. (2016)

# the processed data has the unit of m/yr that was converted from mm/yr
df = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                   names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car', \
                          'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])

In [ ]:
# Decide the weights of the horizontal, vertical components
# f_h, f_v = 1, 1/2
f_h, f_v = 1, 1
# Print the weights of the data
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
# Take the inverse for saving the name of the weights
w_h, w_v = int(1/f_h), int(1/f_v)

# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email

In [ ]:
# meshnameCK = "nicoyaCK3"   # fault zone extended to the whole subduction zone
# meshnameCK = "nicoyaCK4"   # same as CK2, but connecting the trench now

# meshnameCK = "nicoyaCKden_sm"   # based on nicoyaCK3 or 4, but denser mesh size, and smaller fault zone
meshnameCK = "nicoyaCKden_all"   # based on nicoyaCK3 or 4, but denser mesh size, and all subduction interface

def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# string for the homogeneous solution
homo_str = f"_mul{round(mu_expression(mu_b))}u{round(mu_expression(mu_b))}"
# string for the contrast between slab and wedge
sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
# string for the 3d model, value multiplied by 4, relative a reference
# contrast_factor = 4.0  # amplification factor 
contrast_factor = 1.0  # amplification factor 
het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}"

rho_s = 2.5e8   # allows variations of slip of the order of ~15 km, close to the maximum resolution

# L-curve, CK mesh, inversion in HOM model
gammas_CK_hom = [1e2, 4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
outFileName = f"Lcurvelockboth_rs{rho_s:.0e}_{meshnameCK}_{homo_str}.txt"
misfitsCK_hom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])

# L-curve, CK mesh, inversion in SW model
gammas_CK_sw = [1e2, 4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
outFileName = f"Lcurvelockboth_rs{rho_s:.0e}_{meshnameCK}_{sw_str}.txt"
misfitsCK_sw = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])

# L-curve, CK mesh, inversion in 3D model
gammas_CK_3d = [1e2, 4e2, 6e2, 8e2, 1e3, 2e3, 3e3, 4e3, 1e4, 5e4]
outFileName = f"Lcurvelockboth_rs{rho_s:.0e}_{meshnameCK}_{het3d_str}.txt"
misfitsCK_3d = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    names=['d_mis', 'm_mis'])


In [ ]:
print(gammas_CK_hom)
print(gammas_CK_hom[1:])

In [ ]:
# Plot L-curve
fig, axes = plt.subplots(1, 1, figsize=(4,4), dpi=300)

ax = axes
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=8)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=8)
ax.set_title("Interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

color_L = ['silver', 'firebrick', 'white']

ax.plot(misfitsCK_hom['m_mis'][1:], misfitsCK_hom['d_mis'][1:], 
        'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"HOM")
idxCK_hom = 6
ax.plot(misfitsCK_hom.iloc[idxCK_hom]['m_mis'], misfitsCK_hom.iloc[idxCK_hom]['d_mis'], 
        'o', color='k', markerfacecolor='k', markersize=5, 
        label=r"HOM preferred $\gamma$={:.1e}".format(gammas_CK_hom[idxCK_hom]))

ax.plot(misfitsCK_sw['m_mis'][1:], misfitsCK_sw['d_mis'][1:], 
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"SW")
idxCK_sw = 6
ax.plot(misfitsCK_sw.iloc[idxCK_sw]['m_mis'], misfitsCK_sw.iloc[idxCK_sw]['d_mis'], 
        's', color='red', markerfacecolor='red', markersize=5, 
        label=r"SW preferred $\gamma$={:.1e}".format(gammas_CK_sw[idxCK_sw]))

ax.plot(misfitsCK_3d['m_mis'][1:], misfitsCK_3d['d_mis'][1:], 
        'D-', color='blue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"3D")
idxCK_3d = 6
ax.plot(misfitsCK_3d.iloc[idxCK_3d]['m_mis'], misfitsCK_3d.iloc[idxCK_3d]['d_mis'], 
        'D', color='blue', markerfacecolor='blue', markersize=5, 
        label=r"3D preferred $\gamma$={:.1e}".format(gammas_CK_3d[idxCK_3d]))


# for i, gamma in enumerate(gammas_s):
#     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)
# ax.set_ylim(-5, 80)
# ax.set_xlim(0, 0.1)


figpath = datadir + '/figures/locking/'
output_file = figpath + f'Lcurvelockboth_{meshnameCK}.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')

In [ ]:
# # meshnameCK = "nicoyaCK"   # local interface model from C. Kyriakopoulos_etal2015JGRSE
# # meshnameCK = "nicoyaCK2"   # same as above but 5-km mesh size on fault
# meshnameCK = "nicoyaCK3"   # fault zone extended to the whole subduction zone
# # meshnameCK = "nicoyaCK4"   # same as CK2, but connecting the trench now
# print(meshnameCK)

# # meshname = "nicoya"
# # meshname = "nicoya2"   # This has a smaller fault interface
# # meshname = "nicoya3"   # the same as above but 5-km mesh size on fault
# meshname = "nicoya4"   # extended fault area
# print(meshname)

# # if meshnameCK == "nicoyaCK3":   # fault zone extended to the whole subduction zone

# rho_s = 2.5e8   # allows variations of slip of the order of ~15 km, close to the maximum resolution
# gammas_s = [1e2, 2.5e2, 5e2, 7.5e2, 1e3, 2.5e3, 5e3, 1e4, 5e4]

# outFileName = f"Lcurvelockbothrs{rho_s:.0e}{meshnameCK}.txt"
# misfitsCK = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
#                     names=['d_mis', 'm_mis'])
# gammas_s_used = 2.5e3
# idxCK = gammas_s.index(gammas_s_used)
# print(idxCK)
# print(gammas_s[idxCK])

# outFileName = f"Lcurvelockbothrs{rho_s:.0e}{meshname}.txt"
# misfits = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
#                     names=['d_mis', 'm_mis'])
# gammas_s_used = 1e3  
# idx = gammas_s.index(gammas_s_used)
# print(gammas_s[idx])

# ### L-curve criterion ###
# color_L = ['silver', 'firebrick']

# # Plot L-curve
# fig, axes = plt.subplots(1,1,figsize=(4,4), dpi=300)

# # ax = axes[0]
# ax = axes
# ax.set_xlabel(r"$||\mathbf{Gm} - \mathbf{d}||$", fontsize=8)
# ax.set_ylabel(r"$||\mathbf{Lm}||$", fontsize=8)
# ax.set_title("Interseismic locking ratio inversion", fontsize=9)
# ax.tick_params(labelsize=8)

# ax.plot(misfitsCK['d_mis'], misfitsCK['m_mis'], 'o-', color='blue', markerfacecolor=color_L[0],
#         linewidth=1.0, markersize=4, label=r"CK; $\rho$ = {:.1e}".format(rho_s))
# ax.plot(misfitsCK.iloc[idxCK]['d_mis'], misfitsCK.iloc[idxCK]['m_mis'], 's', color='blue', markerfacecolor='blue',
#         markersize=5, label=r"CK; preferred $\gamma$ = {:.1e}".format(gammas_s[idxCK]))
# ax.plot(misfits['d_mis'], misfits['m_mis'], 'o-', color='red', markerfacecolor=color_L[0],
#         linewidth=1.0, markersize=4, label=r"Slab2; $\rho$ = {:.1e}".format(rho_s))
# ax.plot(misfits.iloc[idx]['d_mis'], misfits.iloc[idx]['m_mis'], 's', color='red', markerfacecolor='red',
#         markersize=5, label=r"Slab2; preferred $\gamma$ = {:.1e}".format(gammas_s[idx]))
# # for i, gamma in enumerate(gammas_s):
# #     ax.text(misfits11['d_mis'].iloc[i], misfits11['m_mis'].iloc[i]+0.25, f"{gamma:.0f}", fontsize=6, color='k')
# ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
# ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
# ax.set_box_aspect(1)
# ax.legend(fontsize=8)